# Previsão de risco de atrasos na entrega (Olist Store)
## Preparação dos dados para regressão logística


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_score,
    recall_score, accuracy_score, f1_score, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Bibliotecas importadas com sucesso!")


### 1. Carregamento e Preparação dos Dados
Unificação relacional das tabelas públicas da Olist Store:
- `orders`: pedidos e timestamps de compra, aprovação, despacho e entrega real/estimada.
- `order_items`: valores de produto e frete.
- `customers` e `sellers`: geolocalização e unidades federativas.
- `geolocation`: coordenadas de latitude e longitude dos CEPs brasileiros.
- `order_payments`: modalidades de pagamento e parcelamento.


In [ ]:
orders = pd.read_csv(f'olist_orders_dataset.csv')
items = pd.read_csv(f'olist_order_items_dataset.csv')
customers = pd.read_csv(f'olist_customers_dataset.csv')
sellers = pd.read_csv(f'olist_sellers_dataset.csv')
payments = pd.read_csv(f'olist_order_payments_dataset.csv')
products = pd.read_csv(f'olist_products_dataset.csv')
geo = pd.read_csv(f'olist_geolocation_dataset.csv')

print(f"Total de pedidos registrados: {orders.shape[0]:,}")


In [ ]:
# Filtragem de pedidos concluídos (delivered)
delivered = orders[orders['order_status'] == 'delivered'].dropna(
    subset=['order_delivered_customer_date', 'order_estimated_delivery_date', 'order_purchase_timestamp']
).copy()

# Conversão de datas
delivered['order_purchase_timestamp'] = pd.to_datetime(delivered['order_purchase_timestamp'])
delivered['order_delivered_customer_date'] = pd.to_datetime(delivered['order_delivered_customer_date'])
delivered['order_estimated_delivery_date'] = pd.to_datetime(delivered['order_estimated_delivery_date'])
delivered['order_approved_at'] = pd.to_datetime(delivered['order_approved_at'])

# Variável Alvo Primária (Classificação): 1 = Atraso, 0 = No Prazo
delivered['is_late'] = (delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']).astype(int)

# Variável Alvo Secundária (Regressão): Dias exatos de atraso
delivered['delay_days'] = (delivered['order_delivered_customer_date'] - delivered['order_estimated_delivery_date']).dt.total_seconds() / (24 * 3600)

print(f"Total de pedidos entregues analisados: {len(delivered):,}")
print("Distribuição da variável alvo (is_late):")
print(delivered['is_late'].value_counts())
print(f"Percentual de atrasos na base real: {delivered['is_late'].mean()*100:.2f}% (Forte Desbalanceamento)")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = delivered['is_late'].value_counts()
counts.plot(kind='bar', color=['#2b5c8f', '#d9534f'], ax=ax)
ax.set_title('Desbalanceamento de Classes (Atrasos na Entrega Olist)', fontweight='bold')
ax.set_xticklabels(['No Prazo (0)', 'Atrasado (1)'], rotation=0)
ax.set_ylabel('Quantidade de Encomendas')
for p in ax.patches:
    ax.annotate(f"{p.get_height():,} ({p.get_height()/len(delivered)*100:.1f}%)",
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.show()


### 2. Engenharia de Atributos (Feature Engineering)
Criação de métricas preditivas conhecidas no ciclo de atendimento:
- `estimated_days`: Prazo total prometido ao cliente.
- `approval_time_hours`: Tempo para aprovação financeira do pagamento.
- `distance_km`: Distância geodésica (fórmula de Haversine) entre comprador e vendedor.
- `freight_ratio`: Proporção do frete em relação ao valor da mercadoria.
- `same_state`: Indicador booleano se o vendedor e o comprador estão no mesmo estado.
- `mean_product_weight_g`: Peso médio dos itens (impacto na complexidade do transporte rodoviário).


In [ ]:
def haversine_np(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6367 * c

# Coordenadas médias por prefixo de CEP
geo_clean = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lng=('geolocation_lng', 'mean')
).reset_index()

# Atributos temporais
delivered['estimated_days'] = (delivered['order_estimated_delivery_date'] - delivered['order_purchase_timestamp']).dt.total_seconds() / (24 * 3600)
delivered['approval_time_hours'] = (delivered['order_approved_at'] - delivered['order_purchase_timestamp']).dt.total_seconds() / 3600
delivered['purchase_dayofweek'] = delivered['order_purchase_timestamp'].dt.dayofweek
delivered['purchase_month'] = delivered['order_purchase_timestamp'].dt.month

# Agregação de itens e frete
items_prod = items.merge(products, on='product_id', how='left')
items_seller = items_prod.merge(sellers[['seller_id', 'seller_state', 'seller_zip_code_prefix']], on='seller_id', how='left')

order_items_agg = items_seller.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    items_count=('order_item_id', 'count'),
    mean_product_weight_g=('product_weight_g', 'mean'),
    seller_state=('seller_state', 'first'),
    seller_zip=('seller_zip_code_prefix', 'first')
).reset_index()

# Agregação de pagamentos
payments_agg = payments.groupby('order_id').agg(
    payment_installments=('payment_installments', 'max'),
    payment_type=('payment_type', lambda x: x.mode()[0] if not x.empty else 'credit_card')
).reset_index()

# Junção principal
df = delivered.merge(customers[['customer_id', 'customer_state', 'customer_zip_code_prefix']], on='customer_id', how='left')
df = df.merge(order_items_agg, on='order_id', how='inner')
df = df.merge(payments_agg, on='order_id', how='inner')

# Distância física
df = df.merge(geo_clean.rename(columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix', 'lat': 'cust_lat', 'lng': 'cust_lng'}), on='customer_zip_code_prefix', how='left')
df = df.merge(geo_clean.rename(columns={'geolocation_zip_code_prefix': 'seller_zip', 'lat': 'sell_lat', 'lng': 'sell_lng'}), on='seller_zip', how='left')

df['distance_km'] = haversine_np(df['cust_lng'], df['cust_lat'], df['sell_lng'], df['sell_lat'])
df['distance_km'] = df['distance_km'].fillna(df['distance_km'].median())
df['approval_time_hours'] = df['approval_time_hours'].fillna(df['approval_time_hours'].median())
df['mean_product_weight_g'] = df['mean_product_weight_g'].fillna(df['mean_product_weight_g'].median())

df['same_state'] = (df['customer_state'] == df['seller_state']).astype(int)
df['freight_ratio'] = df['total_freight'] / (df['total_price'] + 1e-5)

print(f"Base consolidada com engenharia de atributos: {df.shape}")


### 3. Divisão dos dados e normalização
Separação estratificada de treino e teste e padronização para regressão logística.


In [ ]:
features = [
    'estimated_days', 'approval_time_hours', 'purchase_dayofweek', 'purchase_month',
    'total_price', 'total_freight', 'items_count', 'mean_product_weight_g',
    'distance_km', 'same_state', 'freight_ratio', 'payment_installments'
]

# Construção do cenário de monitoramento de risco com subamostragem estratificada
# Para alinhamento com as metas operacionais de intervenção do SAC/WMS
df_late = df[df['is_late'] == 1]
df_ontime = df[df['is_late'] == 0]

# Proporção balanceada de contingência (1:1)
df_study = pd.concat([df_late, df_ontime.sample(n=len(df_late), random_state=42)]).sample(frac=1, random_state=42).reset_index(drop=True)

X = df_study[features]
y = df_study['is_late'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Tamanho do conjunto de Treino: {X_train.shape[0]:,} amostras")
print(f"Tamanho do conjunto de Teste: {X_test.shape[0]:,} amostras")
